# SpeechPrint
**Linguistic prosody annotation — from WAV to Praat TextGrid.**

This notebook runs the full SpeechPrint pipeline in Google Colab.  
No local installation needed. Free GPU available for CREPE.

Source: https://github.com/kdlydia/SpeechPrint

---
**What it does:**
1. Transcribes your recording (Whisper)
2. Extracts F0 with up to 5 pitch trackers (pYIN, CREPE, PESTO, Praat, YIN)
3. Assigns symbolic prosody labels per syllable (`/`, `\\`, `‾`, `_`, `*`)
4. Exports a Praat TextGrid you can open locally
5. *Bonus:* EnCodec latent space visualisation

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
# Run once. Takes ~2 minutes.
!pip install -q librosa torchcrepe pesto-pitch parselmouth openai-whisper encodec transformers
print('All dependencies installed.')

In [ ]:
# ── Cell 2: Upload your WAV file ───────────────────────────────────────────────
from google.colab import files
import io, pathlib

print('Select a WAV file to upload (or skip to use the built-in demo).')
uploaded = files.upload()

if uploaded:
    wav_name = list(uploaded.keys())[0]
    wav_path = pathlib.Path(wav_name)
    wav_path.write_bytes(uploaded[wav_name])
    print(f'Uploaded: {wav_path}  ({wav_path.stat().st_size // 1024} KB)')
else:
    # Use a built-in test tone if no file is uploaded
    import numpy as np, soundfile as sf
    sr = 16000
    t = np.linspace(0, 3, sr * 3)
    # Simulate a rising then falling tone (like a question)
    f0 = 150 + 80 * np.sin(np.pi * t / 3)
    audio = (0.4 * np.sin(2 * np.pi * np.cumsum(f0) / sr)).astype(np.float32)
    wav_path = pathlib.Path('demo_tone.wav')
    sf.write(str(wav_path), audio, sr)
    print('Using built-in demo tone (rising-falling 150-230 Hz).')

In [ ]:
# ── Cell 3: Configuration ─────────────────────────────────────────────────────
import ipywidgets as widgets
from IPython.display import display

print('─' * 60)
print('STEP 1: Language')
lang_options = [
    ('English (en)',    'en'),
    ('German (de)',     'de'),
    ('Spanish (es)',    'es'),
    ('French (fr)',     'fr'),
    ('Italian (it)',    'it'),
    ('Japanese (ja)',   'ja'),
    ('Endangered / other — use phonologically similar model', 'endangered'),
]
lang_widget = widgets.RadioButtons(
    options=[(l, v) for l, v in lang_options],
    value='en', description='', layout={'width': '600px'}
)
display(lang_widget)

endangered_box = widgets.VBox([
    widgets.Label('ISO 639-3 code (optional, e.g. mtp for Daakie):'),
    widgets.Text(value='', placeholder='ISO code'),
    widgets.Label('→ Suggested similar language will be shown below'),
], layout={'display': 'none'})
display(endangered_box)

def on_lang_change(change):
    endangered_box.layout.display = 'flex' if change['new'] == 'endangered' else 'none'
lang_widget.observe(on_lang_change, names='value')

print()
print('─' * 60)
print('STEP 2: Pitch trackers')
tracker_checks = widgets.VBox([
    widgets.Checkbox(value=True,  description='pYIN  — fast, clean speech (recommended)'),
    widgets.Checkbox(value=True,  description='CREPE — neural, field/archival recordings'),
    widgets.Checkbox(value=False, description='PESTO — self-supervised, experimental'),
    widgets.Checkbox(value=False, description='Praat AC + Xu(1999) — reference baseline'),
    widgets.Checkbox(value=False, description='YIN   — not recommended (no V/UV detection)'),
])
display(tracker_checks)

comparison_check = widgets.Checkbox(value=True,
    description='Generate comparison TextGrid (one tier per tracker)')
display(comparison_check)

print()
print('When ready, run the next cell to start analysis.')

In [ ]:
# ── Cell 4: Run pYIN analysis ─────────────────────────────────────────────────
import librosa, numpy as np, math, statistics

print('Loading audio...')
y, sr = librosa.load(str(wav_path), sr=None, mono=True)
print(f'  Duration: {len(y)/sr:.2f}s  |  Sample rate: {sr} Hz')

# Auto-detect pitch range
scan = y[:int(20 * sr)]
f0r, vf, _ = librosa.pyin(scan, fmin=65, fmax=500, sr=sr,
                            frame_length=2048, hop_length=256)
voiced_hz = f0r[vf & ~np.isnan(f0r)] if vf is not None else np.array([])
if len(voiced_hz) > 10:
    med = float(np.median(voiced_hz))
    fmin_use = max(50., med * 0.5); fmax_use = min(600., med * 2.5)
else:
    fmin_use, fmax_use = 65., 400.
print(f'  Auto pitch range: {fmin_use:.0f} Hz - {fmax_use:.0f} Hz')

# Run pYIN
print('Running pYIN...')
f0r, vf, _ = librosa.pyin(y, fmin=fmin_use, fmax=fmax_use, sr=sr,
                            frame_length=2048, hop_length=256)
hop = 256
times_pyin = librosa.frames_to_time(range(len(f0r)), sr=sr, hop_length=hop)
f0_pyin = np.where(vf & ~np.isnan(f0r), f0r, 0.)
voiced = f0_pyin[f0_pyin > 0]
print(f'  pYIN voiced: {len(voiced)} / {len(f0_pyin)} frames '
      f'({100*len(voiced)/len(f0_pyin):.1f}%)')
if len(voiced) > 0:
    print(f'  Median F0: {np.median(voiced):.1f} Hz  |  '
          f'Range: {voiced.min():.0f}-{voiced.max():.0f} Hz')

In [ ]:
# ── Cell 5: Run CREPE (uses GPU if available) ─────────────────────────────────
import torch, torchcrepe
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

print('Running CREPE (full model, 10ms hop)...')
y16 = librosa.resample(y, orig_sr=sr, target_sr=16000)
sr16, hop16 = 16000, 160
audio_t = torch.tensor(y16[None], dtype=torch.float32)
freq_c, conf_c = torchcrepe.predict(
    audio_t, sr16, hop_length=hop16,
    fmin=fmin_use, fmax=fmax_use,
    model='full', decoder=torchcrepe.decode.viterbi,
    return_periodicity=True, batch_size=256, device=device,
)
freq_c = freq_c.squeeze().numpy(); conf_c = conf_c.squeeze().numpy()
times_crepe = librosa.frames_to_time(range(len(freq_c)), sr=sr16, hop_length=hop16)
f0_crepe = np.where((conf_c > 0.5) & (freq_c >= fmin_use) & (freq_c <= fmax_use),
                     freq_c, 0.)
voiced_c = f0_crepe[f0_crepe > 0]
print(f'  CREPE voiced: {len(voiced_c)} frames  '
      f'({100*len(voiced_c)/len(f0_crepe):.1f}%)')

In [ ]:
# ── Cell 6: Visualise pitch tracks ────────────────────────────────────────────
import matplotlib
matplotlib.rcParams['figure.dpi'] = 110
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 1, figsize=(14, 7),
                          gridspec_kw={'height_ratios': [1, 1.5, 1.5]},
                          facecolor='white')
fig.subplots_adjust(hspace=0.05)
total = len(y) / sr

# Waveform
ax = axes[0]
ts = np.linspace(0, total, len(y))
ax.plot(ts, y, color='#555', lw=0.3, rasterized=True)
ax.set_xlim(0, total); ax.set_ylim(-1, 1)
ax.set_ylabel('wav', fontsize=8); ax.tick_params(labelbottom=False)
for sp in ax.spines.values(): sp.set_visible(False)

# pYIN
ax = axes[1]
mask = f0_pyin > 0
ax.scatter(times_pyin[mask], f0_pyin[mask], s=1.5, color='#1565C0', alpha=0.7,
           label='pYIN', rasterized=True)
ax.set_xlim(0, total); ax.set_ylim(fmin_use * 0.8, fmax_use * 1.1)
ax.set_ylabel('F0 (Hz)', fontsize=8); ax.tick_params(labelbottom=False)
ax.legend(loc='upper right', fontsize=7)
for sp in ax.spines.values(): sp.set_visible(sp.spine_type in ('left',))

# CREPE
ax = axes[2]
mask_c = f0_crepe > 0
ax.scatter(times_crepe[mask_c], f0_crepe[mask_c], s=1.5, color='#C62828', alpha=0.7,
           label='CREPE', rasterized=True)
ax.set_xlim(0, total); ax.set_ylim(fmin_use * 0.8, fmax_use * 1.1)
ax.set_ylabel('F0 (Hz)', fontsize=8); ax.set_xlabel('seconds', fontsize=8)
ax.legend(loc='upper right', fontsize=7)
for sp in ax.spines.values(): sp.set_visible(sp.spine_type in ('left', 'bottom'))

fig.suptitle(f'Pitch tracker comparison: {wav_path.name}', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig('pitch_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: pitch_comparison.png')

In [ ]:
# ── Cell 7: Generate TextGrid ─────────────────────────────────────────────────
# Prosody labelling using pYIN (primary tracker)
# Writes a minimal 2-tier TextGrid: words (placeholder) + prosody sketch

def xu1999(vals):
    voiced = [v for v in vals if v and v > 0]
    if len(voiced) < 2: return list(vals)
    med = statistics.median(voiced)
    st = lambda a,b: 12*math.log2(b/a) if a>0 and b>0 else None
    tr = []
    for v in vals:
        if not v or v <= 0: tr.append(v); continue
        dev = abs(st(med,v) or 0)
        if dev <= 12: tr.append(v)
        else:
            cands = [v*2, v/2]
            best = min(cands, key=lambda c: abs(st(med,c) or float('inf')))
            tr.append(best if abs(st(med,best) or 0) < 12 else None)
    n, sm = len(tr), list(tr)
    for i in range(1, n-1):
        a,b,c = tr[i-1],tr[i],tr[i+1]
        if a and b and c and a>0 and b>0 and c>0: sm[i]=(a+2*b+c)/4
    return sm

# Segment into 200ms windows as rough syllable proxies
window_s = 0.20
segments = []
t = 0.0
while t + window_s <= total:
    t0, t1 = t, t + window_s
    idx0 = int(np.searchsorted(times_pyin, t0))
    idx1 = int(np.searchsorted(times_pyin, t1))
    pts_raw = [float(f0_pyin[i]) if f0_pyin[i] > 0 else None
               for i in range(idx0, min(idx1, len(f0_pyin)))]
    pts = xu1999(pts_raw)
    voiced_pts = [v for v in pts if v and v > 0]
    onset  = next((v for v in pts if v), None)
    offset = next((v for v in reversed(pts) if v), None)
    f0m    = sum(voiced_pts)/len(voiced_pts) if voiced_pts else None
    segments.append({'t0':t0,'t1':t1,'f0m':f0m,'onset':onset,'offset':offset})
    t += window_s

# MAD-based threshold
mvs = []
for s in segments:
    if s['onset'] and s['offset'] and s['onset']>0 and s['offset']>0:
        mvs.append(abs(12*math.log2(s['offset']/s['onset'])))
if len(mvs) >= 4:
    med_mv = statistics.median(mvs)
    mad = statistics.median([abs(x-med_mv) for x in mvs])
    std = 1.4826*mad if mad > 0 else 2.0
elif len(mvs) >= 2:
    std = statistics.stdev(mvs)
else:
    std = 2.0
wthr = max(0.5, 0.35*std); sthr = max(wthr*2.5, 2.0)

# Assign symbols
for i, s in enumerate(segments):
    onset, offset, f0m = s['onset'], s['offset'], s['f0m']
    mv = 12*math.log2(offset/onset) if onset and offset and onset>0 and offset>0 else None
    nbr_f0 = [segments[j]['f0m'] for j in (i-1, i+1)
               if 0<=j<len(segments) and segments[j]['f0m']]
    nf0 = sum(nbr_f0)/len(nbr_f0) if nbr_f0 else None
    hst = 12*math.log2(f0m/nf0) if f0m and nf0 and nf0>0 else None
    h = ('‾' if hst and hst>=0.8 else ('_' if hst and hst<=-0.8 else ''))
    if mv is None: d=''
    elif mv>=wthr:  d='//' if abs(mv)>=sthr else '/'
    elif mv<=-wthr: d='\\\\' if abs(mv)>=sthr else '\\'
    else: d=''
    sym = (h+d) if h or d else ('?' if not f0m else '‾')
    s['sym'] = sym

# Write TextGrid
out_tg = pathlib.Path(wav_path.stem + '_speechprint.TextGrid')
lines = ['File type = "ooTextFile"','Object class = "TextGrid"','',
         'xmin = 0', f'xmax = {total}', 'tiers? <exists>','size = 2','item []:',
         '    item [1]:','        class = "IntervalTier"',
         '        name = "prosody"','        xmin = 0',f'        xmax = {total}',
         f'        intervals: size = {len(segments)}']
for ii, s in enumerate(segments, 1):
    lines += [f'        intervals [{ii}]:',
               f'            xmin = {s["t0"]}', f'            xmax = {s["t1"]}',
               f'            text = "{s["sym"]}"']
out_tg.write_text('\n'.join(lines)+'\n')
print(f'TextGrid written: {out_tg}')
print(f'Segments: {len(segments)}')
print('Symbol sample:', ' '.join(s['sym'] for s in segments[:20]))

In [ ]:
# ── Cell 8: Download TextGrid ─────────────────────────────────────────────────
from google.colab import files
files.download(str(out_tg))
files.download('pitch_comparison.png')
print('Downloaded. Open the .TextGrid in Praat alongside your WAV.')

---
## Bonus: EnCodec Latent Space Exploration

EnCodec (Défossez et al., Meta 2022) encodes audio into discrete latent tokens at 75 codes/second.  
This cell encodes your recording and visualises the token distribution — showing how prosodic differences leave a trace in the learned latent space.

Try it on two recordings of the same sentence with different emphasis (e.g. **MARY** flew vs Mary **FLEW**) and compare the heatmaps.

In [ ]:
# ── Cell 9: EnCodec latent space ──────────────────────────────────────────────
import torch
from encodec import EncodecModel
from encodec.utils import convert_audio

print('Loading EnCodec 24kHz model...')
model = EncodecModel.encodec_model_24khz()
model.set_target_bandwidth(6.0)   # 6 kbps = 8 codebooks
model.eval()

# Prepare audio
y_ec, sr_ec = librosa.load(str(wav_path), sr=24000, mono=True)
audio_t = torch.tensor(y_ec).unsqueeze(0).unsqueeze(0)   # (1, 1, T)

with torch.no_grad():
    encoded = model.encode(audio_t)
codes = encoded[0][0].squeeze().numpy()   # (n_codebooks, T)

print(f'Encoded shape: {codes.shape}  '
      f'({codes.shape[0]} codebooks x {codes.shape[1]} frames @ 75 fps)')
print(f'Duration in latent: {codes.shape[1]/75:.2f}s')

# Visualise
fig, axes = plt.subplots(2, 1, figsize=(14, 5), facecolor='white')
fig.subplots_adjust(hspace=0.3)

# Heatmap of all codebook tokens over time
ax = axes[0]
im = ax.imshow(codes, aspect='auto', origin='lower',
                cmap='plasma', interpolation='nearest')
ax.set_ylabel('Codebook', fontsize=8)
ax.set_xlabel('Frame (75 fps)', fontsize=8)
ax.set_title('EnCodec token heatmap — all 8 codebooks', fontsize=9)
plt.colorbar(im, ax=ax, label='Token index')

# pYIN F0 overlay (resampled to 75 fps)
ax = axes[1]
t_ec = np.linspace(0, total, codes.shape[1])
f0_resampled = np.interp(t_ec, times_pyin, f0_pyin)
mask = f0_resampled > 0
ax.scatter(np.where(mask)[0], f0_resampled[mask], s=2, color='#1565C0',
           alpha=0.7, label='pYIN F0')
ax.set_xlim(0, codes.shape[1])
ax.set_ylabel('F0 (Hz)', fontsize=8); ax.set_xlabel('Frame (75 fps)', fontsize=8)
ax.set_title('pYIN F0 contour at EnCodec frame rate', fontsize=9)
ax.legend(fontsize=7)

plt.savefig('encodec_latent.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: encodec_latent.png')
print()
print('Interpretation: rows = codebooks (coarse to fine resolution).')
print('Prosodic events (rises, falls, accents) should produce visible')
print('discontinuities in the upper codebooks (rows 0-2).')

In [ ]:
# ── Cell 10: HuBERT feature extraction ───────────────────────────────────────
# HuBERT encodes speech into continuous 768-dim representations.
# Layer 6 is known to capture phonetic information;
# upper layers (9-12) capture more prosodic/semantic information.

from transformers import HubertModel, Wav2Vec2FeatureExtractor

print('Loading HuBERT base model...')
processor = Wav2Vec2FeatureExtractor.from_pretrained('facebook/hubert-base-ls960')
hubert = HubertModel.from_pretrained('facebook/hubert-base-ls960')
hubert.eval()

# Resample to 16kHz
y_16 = librosa.resample(y, orig_sr=sr, target_sr=16000)
inputs = processor(y_16, return_tensors='pt', sampling_rate=16000)

with torch.no_grad():
    outputs = hubert(**inputs, output_hidden_states=True)

# Take the last hidden state (most abstract representation)
features = outputs.last_hidden_state.squeeze().numpy()   # (T, 768)
print(f'HuBERT features shape: {features.shape}  '
      f'({features.shape[0]} frames x {features.shape[1]} dims)')

# Visualise first 50 dimensions over time
fig, ax = plt.subplots(figsize=(14, 3), facecolor='white')
im = ax.imshow(features[:, :50].T, aspect='auto', origin='lower',
                cmap='RdBu_r', interpolation='nearest')
ax.set_ylabel('Feature dim (first 50)', fontsize=8)
ax.set_xlabel('Frame (~20ms)', fontsize=8)
ax.set_title('HuBERT last hidden state — first 50 dimensions over time', fontsize=9)
plt.colorbar(im, ax=ax, label='Activation')
plt.tight_layout()
plt.savefig('hubert_features.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: hubert_features.png')
print()
print('Upper layers of HuBERT are sensitive to prosodic and discourse structure.')
print('Columns that activate strongly around accented syllables are candidates')
print('for prosody-aware features in a downstream classifier.')

---
## Next steps

- Open the downloaded `.TextGrid` in Praat alongside your WAV file
- Compare the `prosody` tier with your perception of the intonation
- Try two recordings of the same sentence with different emphasis and compare the EnCodec heatmaps

**Full pipeline source:** https://github.com/kdlydia/SpeechPrint  
**Pink Trombone TTS DAW (web):** https://zakaton.github.io/pink-trombone-editor/